In [ ]:
import os, sys

PROJECT_ROOT = '/scratch/jq2uw/derm_vlms'
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from eval.metrics import eval_model, compute_metrics

VLMS = {
    'DermatoLlama': 'results/dermato_llama_predictions_all.csv',
    'MedGemma':     'results/medgemma_predictions_all.csv',
    'GPT-5.3':      'results/gpt53_predictions_all.csv',
}

dfs = {}
metrics = {}
for name, path in VLMS.items():
    full = os.path.join(PROJECT_ROOT, path)
    if not os.path.exists(full):
        print(f'[SKIP] {name}: {path} not found')
        continue
    df = pd.read_csv(full)
    df = eval_model(df)
    dfs[name] = df
    metrics[name] = compute_metrics(df)
    print(f'{name}: {len(df)} rows loaded')

print(f'\nLoaded {len(dfs)} models')

In [ ]:
# Table 1: Overall top-1 / top-3 accuracy
rows = []
for name, m in metrics.items():
    ov = m['overall'].iloc[0]
    rows.append({
        'Model': name,
        'N': int(ov['n']),
        'Parsed (%)': f"{ov['parsed'] / ov['n'] * 100:.1f}",
        'Matched (%)': f"{ov['matched'] / ov['n'] * 100:.1f}",
        'Top-1 Acc (%)': f"{ov['top1_acc'] * 100:.1f}",
        'Top-3 Acc (%)': f"{ov['top3_acc'] * 100:.1f}",
    })
table1 = pd.DataFrame(rows)
print('=== Overall Accuracy ===')
table1

In [ ]:
# Table 2: Accuracy by image_mode
frames = []
for name, m in metrics.items():
    t = m['by_image_mode'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table2 = pd.concat(frames, ignore_index=True)
table2['top1_acc'] = (table2['top1_acc'] * 100).round(1)
table2['top3_acc'] = (table2['top3_acc'] * 100).round(1)
table2 = table2.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N'})
print('=== Accuracy by Image Mode ===')
table2

In [ ]:
# Table 3: Accuracy by y3 class
frames = []
for name, m in metrics.items():
    t = m['by_y3'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table3 = pd.concat(frames, ignore_index=True)
table3['top1_acc'] = (table3['top1_acc'] * 100).round(1)
table3['top3_acc'] = (table3['top3_acc'] * 100).round(1)
table3 = table3.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N', 'ground_truth': 'y3'})
print('=== Accuracy by y3 Class ===')
table3

In [ ]:
# Table 4: Accuracy by y16 class
frames = []
for name, m in metrics.items():
    t = m['by_y16'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table4 = pd.concat(frames, ignore_index=True)
table4['top1_acc'] = (table4['top1_acc'] * 100).round(1)
table4['top3_acc'] = (table4['top3_acc'] * 100).round(1)
table4 = table4.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N'})
print('=== Accuracy by y16 Diagnosis ===')
table4

In [ ]:
# Table 5: Unmatched diagnosis terms (for refining synonym dictionary)
for name, m in metrics.items():
    um = m['unmatched']
    print(f'\n=== {name}: {len(um)} unmatched terms ===')
    if not um.empty:
        print(um['unmatched_term'].value_counts().head(20))

In [ ]:
# Save consolidated results
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
table1.to_csv(os.path.join(RESULTS_DIR, 'eval_overall.csv'), index=False)
table4.to_csv(os.path.join(RESULTS_DIR, 'eval_by_y16.csv'), index=False)
print('Saved eval_overall.csv and eval_by_y16.csv')